# Part Two: Data Cleaning
Every rule is *fitted on the training data only* (frequent labels, modes, medians) and then applied unchanged to
the test data, so the Bonus-task model sees exactly the same cleaning and nothing leaks from the test set.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

TEAM = "llcd"
TRAIN = Path("dataset/Task2/train.csv")
TEST = Path("dataset/Bonus_Task/test.csv")

MIN_COUNT = 20      # a label rarer than this in the training data is merged into "other"
AGREE = 0.95        # use the atmosphere string to fill a gas column only if it matches the column this often
IMPUTE_FLAG = 0.05  # add an `<col>_imputed` flag column when more than 5% of a column had to be imputed

raw_train, raw_test = pd.read_csv(TRAIN), pd.read_csv(TEST)

## 1. Audit the raw data

In [2]:
print("missing values per column (train):")
print(raw_train.isna().sum()[lambda s: s > 0].to_string())
sfx = raw_train.planet_name.str.extract(r"_(\d+)$", expand=False).astype(float)
print("\nplanet_name suffix != planetId:", int((sfx != raw_train.planetId).sum()),
      "| duplicate rows:", int(raw_train.duplicated().sum()), "| duplicate planetId:", int(raw_train.planetId.duplicated().sum()))
for c in ["star_type", "planet_type", "water_presence", "surface_composition", "rings", "primary_hazard"]:
    low = raw_train[c].str.strip().str.lower().nunique()
    print(f"{c}: {raw_train[c].nunique()} labels, {low} after lower-casing alone")

missing values per column (train):
star_type                 1
orbital_distance_au     637
orbital_period_days     639
day_length_hours         56
axial_tilt_deg           48
escape_velocity_kms     642
oxygen_percent          972
co2_percent            1210
methane_percent         882
water_presence            4
moons_count               7
rings                     7

planet_name suffix != planetId: 11 | duplicate rows: 0 | duplicate planetId: 0
star_type: 51 labels, 43 after lower-casing alone
planet_type: 44 labels, 34 after lower-casing alone
water_presence: 15 labels, 15 after lower-casing alone
surface_composition: 59 labels, 59 after lower-casing alone
rings: 4 labels, 4 after lower-casing alone
primary_hazard: 43 labels, 43 after lower-casing alone


## 2. Label-cleaning rules
Spelling, case and ordering variants are merged. Anything that is still a rare one-off afterwards (a free-text
description such as "ice world with subsurface ocean") is not a reliable category and becomes `other`.

In [3]:
STAR = {"red dwarf": "m-type red dwarf", "m-class red dwarf": "m-type red dwarf", "red dwarf (m-type)": "m-type red dwarf",
        "m-type main sequence": "m-type red dwarf", "white dwarf": "white dwarf remnant", "brown dwarf": "brown dwarf companion",
        "orange dwarf": "k-type orange dwarf", "m-type subdwarf": "subdwarf"}
SURFACE = {"icy": "ice", "rock": "rocky", "basaltic": "volcanic", "basalt": "volcanic"}
HAZARD = {"seismic activity": "seismic", "seismic_activity": "seismic", "extreme seismic activity": "seismic",
          "extreme_seismic": "seismic", "frequent_storms": "storms", "seasonal_storms": "storms", "seasonal storms": "storms",
          "thermal vents": "thermal_vents", "predatory megafauna": "predators", "temperature_extremes": "extreme_temperature",
          "extreme_temperature_swings": "extreme_temperature", "extreme_temperature_differential": "extreme_temperature",
          "extreme_temperature_gradient": "extreme_temperature"}


def star(v):          # "G-type main sequence + M-type red dwarf binary" -> "binary system (g+m)"
    letters = re.findall(r"([afgkm])-type", v)
    return f"binary system ({'+'.join(sorted(letters))})" if len(letters) == 2 and not v.startswith("close") else STAR.get(v, v)


def planet(v):        # "tidally locked ocean world" -> "tidally locked"
    return "tidally locked" if v.startswith("tidally locked") else {"relic world": "relic/artificial world"}.get(v, v)


def surface(v):       # "rocky+ice" == "ice+rocky" == "icy rocky"
    return "+".join(sorted({SURFACE.get(p, p) for p in re.split(r"[+ ]", v)}))


def sorted_parts(mapping):    # compound labels: "seas+subsurface" == "subsurface+seas"
    return lambda v: "+".join(sorted(mapping.get(p, p) for p in v.split("+")))


RULES = {"star_type": star, "planet_type": planet, "surface_composition": surface, "water_presence": sorted_parts({}),
         "primary_hazard": sorted_parts(HAZARD), "rings": lambda v: {"true": "yes", "false": "no"}.get(v, v)}
KEEP = {"none"}       # a rare label that is meaningful on its own (primary_hazard = none)
BOUNDS = {c: (0, 100) for c in ["oxygen_percent", "co2_percent", "methane_percent", "cloud_coverage_percent", "rare_mineral_index"]}
BOUNDS |= {"albedo": (0, 1), "axial_tilt_deg": (0, 180), "mean_temp_c": (-273.15, np.inf), "moons_count": (0, np.inf),
           "temp_range_c": (0, np.inf)}
BOUNDS |= {c: (1e-9, np.inf) for c in ["orbital_distance_au", "orbital_period_days", "day_length_hours", "gravity_g",
                                       "escape_velocity_kms", "atmospheric_pressure_atm"]}
GAS = {"oxygen_percent": "O2", "co2_percent": "CO2", "methane_percent": "CH4"}
NUM = [c for c in raw_train.select_dtypes("number") if c not in ("planetId", "settlement_index")]

## 3. The cleaning function
1. **Labels:** lower-case, merge variants, merge rare one-offs into `other`, fill the few missing labels with the most common one.
2. **Impossible values** (percentages outside 0-100, negative pressure, below absolute zero...) are treated as corrupted -> missing.
3. **Atmosphere string vs numeric gas columns:** `"N2 90% O2 8% Ar 2%"` states the O2 percentage, so a missing
   `oxygen_percent` is filled from it (only if the two agree on the training data). If the gas is *not* listed, its
   typical trace value among such planets is used.
4. **Remaining missing numbers:** training median (robust to the heavy tails), plus an `_imputed` flag for the columns
   where more than 5% were missing, so the information "this value was absent" is not lost.

In [4]:
def tidy(col, fn):
    s = col.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    return s.map(fn, na_action="ignore")


def clean(df, ref=None):
    df, fit = df.copy(), ref is None
    ref = {} if fit else ref
    for c, fn in RULES.items():                                                    # 1. labels
        s = tidy(df[c], fn)
        if fit:
            vc = s.value_counts()
            ref[c] = (set(vc[vc >= MIN_COUNT].index) | KEEP, vc.index[0])
        keep, mode = ref[c]
        df[c] = s.where(s.isin(keep) | s.isna(), "other").fillna(mode)
    bad = {c: int(((df[c] < lo) | (df[c] > hi)).sum()) for c, (lo, hi) in BOUNDS.items()}   # 2. impossible values
    for c, (lo, hi) in BOUNDS.items():
        df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
    print("impossible values set to missing:", {k: v for k, v in bad.items() if v} or "none")
    for c, g in GAS.items():                                                       # 3. atmosphere string
        s = df.atmosphere_primary_gases.str.extract(rf"\b{g} (\d+(?:\.\d+)?)%", expand=False).astype(float)
        both = s.notna() & df[c].notna()
        if fit:
            ref[c] = {"agree": np.isclose(s[both], df.loc[both, c], atol=0.5).mean(), "trace": df.loc[s.isna(), c].median()}
            print(f"{c}: string and column agree on {ref[c]['agree']:.1%} of {both.sum()} rows; "
                  f"typical value when gas not listed: {ref[c]['trace']:.2f}")
        if ref[c]["agree"] >= AGREE:
            df[c] = df[c].fillna(s).fillna(ref[c]["trace"])
    na = df[NUM].isna()                                                            # 4. remaining numbers
    if fit:
        ref["median"], ref["flag"] = df[NUM].median(), list(na.columns[na.mean() > IMPUTE_FLAG])
    for c in ref["flag"]:
        df[c + "_imputed"] = na[c].astype(int)
    df[NUM] = df[NUM].fillna(ref["median"])
    df["moons_count"] = df["moons_count"].round().astype(int)
    return df, ref

## 4. Clean the training data, then apply the same rules to the test data

In [5]:
train, ref = clean(raw_train)
test, _ = clean(raw_test, ref)
assert not train.isna().any().any() and not test.isna().any().any()
train.to_csv(f"{TEAM}_cleaned_dataset.csv", index=False)
test.to_csv(f"{TEAM}_cleaned_test.csv", index=False)      # same rules, for the Bonus-task model
print(f"\ntrain {raw_train.shape} -> {train.shape} | test {raw_test.shape} -> {test.shape}")
print("imputation-flag columns:", ref["flag"])

impossible values set to missing: none
oxygen_percent: string and column agree on 99.9% of 2869 rows; typical value when gas not listed: 0.00
co2_percent: string and column agree on 98.4% of 3665 rows; typical value when gas not listed: 0.04
methane_percent: string and column agree on 99.6% of 985 rows; typical value when gas not listed: 0.04
impossible values set to missing: none

train (6400, 31) -> (6400, 34) | test (1600, 30) -> (1600, 33)
imputation-flag columns: ['orbital_distance_au', 'orbital_period_days', 'escape_velocity_kms']


## 5. Before / after

In [6]:
for c in RULES:
    print(f"\n{c}: {raw_train[c].nunique()} -> {train[c].nunique()} labels | rows labelled 'other': train {int((train[c] == 'other').sum())}, "
          f"test {int((test[c] == 'other').sum())}")
    print(train[c].value_counts().head(8).to_dict())
print("\nmost common raw -> clean changes (excluding pure case changes):")
for c in ["star_type", "planet_type", "primary_hazard"]:
    ch = pd.DataFrame({"raw": raw_train[c], "clean": train[c]}).dropna()
    ch = ch[ch.raw.str.lower().str.strip() != ch.clean]
    print(f"\n{c}:\n{ch.value_counts().head(8).to_string()}")
print("\nmissing values after cleaning:", int(train.isna().sum().sum()))


star_type: 51 -> 18 labels | rows labelled 'other': train 21, test 6
{'m-type red dwarf': 795, 'g-type main sequence': 535, 'k-type main sequence': 403, 'close binary (tidal heating)': 397, 'k-type orange dwarf': 396, 'binary system (g+m)': 383, 'f-type main sequence': 360, 'binary system (f+g)': 354}

planet_type: 44 -> 11 labels | rows labelled 'other': train 19, test 7
{'ocean world': 856, 'terrestrial': 757, 'tidally locked': 736, 'ice world': 670, 'relic/artificial world': 659, 'volcanic world': 624, 'super-earth': 556, 'rogue planet': 537}

surface_composition: 59 -> 14 labels | rows labelled 'other': train 52, test 11
{'rocky': 1991, 'volcanic': 918, 'ice': 817, 'crystalline': 799, 'sandy': 713, 'metallic': 706, 'ice+rocky': 157, 'metallic+rocky': 87}

water_presence: 15 -> 6 labels | rows labelled 'other': train 37, test 8
{'subsurface': 1869, 'seas': 1673, 'global ocean': 1406, 'none': 721, 'trace': 694, 'other': 37}

primary_hazard: 43 -> 9 labels | rows labelled 'other': tr